# 👁️ Real-Time Eye Gaze Tracking & Attention Estimation

Welcome to the **Real-Time Eye Gaze Tracking & Attention Estimation** educational walkthrough! This notebook covers the mathematics, machine learning models, and computer vision algorithms behind real-time attention tracking.

## 🌟 Project Applications
- **Educational Technology**: Analyzing student engagement during remote learning lectures.
- **Accessibility Systems**: Controlling mouse cursors and virtual keyboards using gaze direction.
- **Driver Monitoring Systems (DMS)**: Tracking driver alertness and flagging distraction or fatigue.
- **UX/UI Research**: Heatmapping where users focus their gaze on web pages and application layouts.

---  
## 🧠 1. Theoretical Foundations & Mathematics

### A. Facial Landmarks (MediaPipe Face Mesh)
Our pipeline leverages the **MediaPipe Face Mesh Tasks API**, which detects a dense set of **478 3D facial landmarks** in real-time. In addition to standard facial contours, this model includes **refined iris landmark points**:
- **Anatomical Right Eye (Screen Left)**: Landmarks 133 (inner corner), 33 (outer corner), 159 (top), 145 (bottom), and **468** (iris center).
- **Anatomical Left Eye (Screen Right)**: Landmarks 362 (inner corner), 263 (outer corner), 386 (top), 374 (bottom), and **473** (iris center).

### B. Rotation-Invariant Vector Projection
Standard iris-to-eye center calculations ($x / w$) break down when a user tilts their head (roll), because the horizontal axis of the eye shifts relative to the screen axis.

To make our detector **immune to head tilt**, we project the pupil position vector onto the eye's primary horizontal and vertical axes:

Let $\vec{p}_{left}$ and $\vec{p}_{right}$ be the 2D pixel coordinates of the eye corners. The primary horizontal vector is defined as:
$$\vec{u} = \vec{p}_{right} - \vec{p}_{left}$$

Let $\vec{p}_{iris}$ be the 2D pupil center. The vector from the left corner to the pupil is:
$$\vec{v} = \vec{p}_{iris} - \vec{p}_{left}$$

We calculate the horizontal ratio $t_h$ using the mathematical dot product projection:
$$t_h = \frac{\vec{v} \cdot \vec{u}}{\|\vec{u}\|^2}$$

A ratio $t_h \approx 0.5$ indicates the user is looking directly at the center, $t_h < 0.38$ indicates looking to their right (screen-left), and $t_h > 0.62$ indicates looking to their left (screen-right). The same projection technique is applied vertically using top-to-bottom eyelid landmarks.

### C. Head Pose Estimation (Perspective-n-Point)
To estimate 3D head rotation (Pitch, Yaw, Roll), we map 2D image coordinates to a generic 3D human face model. We solve the **Perspective-n-Point (PnP)** problem using OpenCV's `solvePnP`:

$$s \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \begin{bmatrix} R & | & T \end{bmatrix} \begin{bmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{bmatrix}$$

Where:
- $K$ is the camera intrinsic matrix.
- $R$ is the $3 \times 3$ rotation matrix (decomposed into Pitch, Yaw, and Roll).
- $T$ is the translation vector.
- $X_w, Y_w, Z_w$ are the 3D model coordinates (e.g. nose tip at `0,0,0`).

---  
## 🛠️ 2. Interactive Pipeline Setup

Let's import dependencies and verify that the models and visualizations compile correctly.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Set matplotlib inline visualization
get_ipython().run_line_magic('matplotlib', 'inline')

print("[OK] Core computer vision libraries imported successfully!")

### A. Importing Modular Packages
Now we import our clean package classes: `GazeDetector`, `HeadPoseEstimator`, `AttentionClassifier`, and visualization widgets from `utils`.

In [ ]:
from gaze_tracker.detector import GazeDetector
from gaze_tracker.pose_estimator import HeadPoseEstimator
from gaze_tracker.attention_classifier import AttentionClassifier
from gaze_tracker.utils import draw_eye_hud, draw_head_axes, draw_dashboard

print("[OK] Gaze Tracking package components successfully loaded!")

### B. Executing the Tracking Pipeline on Portrait Image
We load the reference portrait face, feed it through the estimators, and plot the resulting HUD overlay directly in the notebook.

In [ ]:
# Load reference portrait image
image_path = "sample_output/portrait_face.png"
img = cv2.imread(image_path)

if img is None:
    raise FileNotFoundError("Sample portrait not found! Please run 'python3 validate_pipeline.py' in the terminal first.")

# 1. Initialize detector, pose estimator, and classifier
detector = GazeDetector()
pose_estimator = HeadPoseEstimator()
classifier = AttentionClassifier(ema_alpha=1.0) # Using alpha=1 for single frame instant update

# 2. Run facial landmark detection
gaze_results = detector.process_frame(img)

if gaze_results['success']:
    # 3. Estimate Head Pose
    landmarks = gaze_results['raw_landmarks']
    pose_results = pose_estimator.estimate(landmarks, img.shape)
    
    # 4. Classify Attention metrics
    attention_results = classifier.update(gaze_results, pose_results)
    
    # 5. Render visualizations onto a copy of the image
    annotated_img = img.copy()
    
    # Draw high-tech target crosshairs on both eyes
    draw_eye_hud(annotated_img, gaze_results['left_eye'], color=(255, 255, 0), crosshair_color=(255, 0, 255))
    draw_eye_hud(annotated_img, gaze_results['right_eye'], color=(255, 255, 0), crosshair_color=(255, 0, 255))
    
    # Project 3D coordinate axes from nose tip
    if pose_results['success']:
        draw_head_axes(annotated_img, pose_results['nose_tip'], pose_results['projected_axis'])
        
    # Overlay the cyber-tech Telemetry HUD dashboard panel
    draw_dashboard(annotated_img, attention_results)
    
    # Convert BGR (OpenCV format) to RGB (Matplotlib format)
    rgb_annotated = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    # 6. Display in Notebook
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb_annotated)
    plt.axis('off')
    plt.title("Real-Time Gaze Tracking Telemetry Output", fontsize=15, fontweight='bold', pad=15)
    plt.show()
    
    # Print textual summary of the output
    print("="*50)
    print(" 👁️  REAL-TIME TELEMETRY STATS SUMMARY")
    print("="*50)
    print(f"  Attention State:  {attention_results['state']}")
    print(f"  Session Focus:    {attention_results['focus_index']:.1f}%")
    print(f"  Head Yaw Angle:   {attention_results['head_pose']['yaw']:.2f} degrees")
    print(f"  Head Pitch Angle: {attention_results['head_pose']['pitch']:.2f} degrees")
    print(f"  Left Eye Gaze H:  {gaze_results['left_eye']['h_ratio']:.3f}")
    print(f"  Right Eye Gaze H: {gaze_results['right_eye']['h_ratio']:.3f}")
    print("="*50)
else:
    print("[ERROR] GazeDetector failed to detect face landmarks. Please check image illumination.")

detector.close()

---  
## 🚀 3. Summary of Core Insights
1. **Mathematical Projection**: By projecting the 2D pupil center onto the eyelid vectors, we make the gaze ratio calculations extremely robust against camera roll or lateral tilts.
2. **Sensor Fusion**: Attention estimation is much more accurate when combining head pose yaw/pitch *with* iris gaze directions, filtering out false distractions (e.g. eyes naturally shifting during centered head discussion).
3. **State Buffering**: Applying an Exponential Moving Average (EMA) and a grace period prevents rapid state-flapping caused by standard blinks or eye tracking jitters.